In [ ]:
!pip install librosa imbalanced-learn scikit-learn


In [ ]:
import os
import glob
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from sklearn.metrics import confusion_matrix, roc_curve
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
def get_paths_labels(folder_path, label):
    file_paths = glob.glob(os.path.join(folder_path, "*.wav"))
    return list(zip(file_paths, [label] * len(file_paths)))

def extract_lfcc(file_path, sr=16000, n_lfcc=20):
    y, _ = librosa.load(file_path, sr=sr)
    # Simulate LFCC by using MFCC — Librosa doesn't have native LFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_lfcc)
    return mfcc.T

def extract_features(dataset):
    features, labels = [], []
    for path, label in dataset:
        try:
            lfcc = extract_lfcc(path)
            features.append(np.mean(lfcc, axis=0))  # mean pooling
            labels.append(label)
        except Exception as e:
            print(f"Error: {path} | {e}")
    return np.array(features), np.array(labels)


In [ ]:
train_real = get_paths_labels("/kaggle/input/scenefake/train/real", 0)
train_fake = get_paths_labels("/kaggle/input/scenefake/train/fake", 1)
test_real  = get_paths_labels("/kaggle/input/scenefake/eval/real", 0)
test_fake  = get_paths_labels("/kaggle/input/scenefake/eval/fake", 1)

train_data = train_real + train_fake
test_data = test_real + test_fake


In [ ]:
X_train, y_train = extract_features(train_data)
X_test, y_test = extract_features(test_data)

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)


In [ ]:
class AudioDataset(data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = data.DataLoader(AudioDataset(X_train_res, y_train_res), batch_size=32, shuffle=True)
test_loader = data.DataLoader(AudioDataset(X_test, y_test), batch_size=32)


In [ ]:
import torch.nn as nn

class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(100):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
model.eval()
all_probs, all_preds, all_targets = [], [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        out = model(xb)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(out, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_targets.extend(yb.numpy())

# Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# Equal Error Rate (EER)
fpr, tpr, thresholds = roc_curve(all_targets, all_probs)
fnr = 1 - tpr
eer_threshold = thresholds[np.nanargmin(np.absolute((fnr - fpr)))]
eer = fpr[np.nanargmin(np.absolute((fnr - fpr)))]

print(f"Equal Error Rate (EER): {eer:.4f}")
